In [ ]:
# skin_cancer_qiea_complete_pipeline.py
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import shap
import cv2
import time
import psutil
import os
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve

# -------------------------------
# Data Preprocessing
# -------------------------------
img_size = 224
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

train_gen = datagen.flow_from_directory(
    "ISIC_dataset/",
    target_size=(img_size, img_size),
    batch_size=batch_size,
    subset="training",
    class_mode="binary"
)

val_gen = datagen.flow_from_directory(
    "ISIC_dataset/",
    target_size=(img_size, img_size),
    batch_size=batch_size,
    subset="validation",
    class_mode="binary"
)

# -------------------------------
# Feature Extraction with MobileNetV2
# -------------------------------
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(img_size, img_size, 3))
base_model.trainable = False

feature_extractor = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D()
])

X_train = feature_extractor.predict(train_gen)
y_train = train_gen.classes
X_val = feature_extractor.predict(val_gen)
y_val = val_gen.classes

# -------------------------------
# Quantum-Inspired Evolutionary Algorithm (QIEA) - Simplified
# -------------------------------
def qiea_feature_selection(X, y, num_features=128):
    variances = np.var(X, axis=0)
    idx = np.argsort(variances)[-num_features:]
    return X[:, idx], idx

X_train_opt, selected_idx = qiea_feature_selection(X_train, y_train)
X_val_opt = X_val[:, selected_idx]

# -------------------------------
# Lightweight CNN Classifier
# -------------------------------
classifier = models.Sequential([
    layers.Input(shape=(X_train_opt.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

classifier.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                   loss="binary_crossentropy",
                   metrics=["accuracy","Precision","Recall"])

start_time = time.time()
history = classifier.fit(X_train_opt, y_train, epochs=10, batch_size=32,
                         validation_data=(X_val_opt, y_val))
inference_time = (time.time() - start_time)/len(X_val_opt)

# -------------------------------
# Model Compression (Pruning + Quantization)
# -------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
open("compressed_model.tflite", "wb").write(tflite_model)
model_size = os.path.getsize("compressed_model.tflite")/1024/1024

# -------------------------------
# Explainability: Grad-CAM
# -------------------------------
def grad_cam(img_array, model, layer_name="Conv_1"):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = np.maximum(heatmap, 0) / np.max(heatmap)
    return heatmap

sample_img, _ = val_gen.next()
heatmap = grad_cam(sample_img, base_model)
plt.imshow(sample_img[0])
plt.imshow(cv2.resize(heatmap.numpy(), (img_size, img_size)), cmap="jet", alpha=0.5)
plt.title("Grad-CAM")
plt.show()

# -------------------------------
# Explainability: SHAP
# -------------------------------
explainer = shap.KernelExplainer(classifier.predict, X_train_opt[:100])
shap_values = explainer.shap_values(X_val_opt[:10])
shap.summary_plot(shap_values, X_val_opt[:10])

# -------------------------------
# Performance Plots
# -------------------------------
# Accuracy & Loss
plt.figure(figsize=(10,5))
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.title("Training Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(10,5))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.title("Training Loss")
plt.legend()
plt.show()

# Confusion Matrix
y_pred = (classifier.predict(X_val_opt) > 0.5).astype("int32")
cm = confusion_matrix(y_val, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_val, classifier.predict(X_val_opt))
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (area = {roc_auc:.2f})")
plt.plot([0,1],[0,1],"--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_val, classifier.predict(X_val_opt))
plt.figure()
plt.plot(recall, precision, label="PR curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

# -------------------------------
# Energy Consumption (Proxy via CPU usage)
# -------------------------------
cpu_usage = psutil.cpu_percent()
plt.bar(["CPU Usage (%)"], [cpu_usage])
plt.title("Energy Consumption Proxy")
plt.show()

# -------------------------------
# Inference Time & Model Size
# -------------------------------
plt.bar(["Inference Time (s)"], [inference_time])
plt.title("Average Inference Time per Sample")
plt.show()

plt.bar(["Model Size (MB)"], [model_size])
plt.title("Compressed Model Size")
plt.show()
